# Graph ML with Neo4j-style Data — Worked Example

This notebook demonstrates how to extract **X** (features) and **y** (labels) from graph data, how to compute the **Laplacian matrix** and use it practically (spectral features), and how to train a simple model to predict node labels (e.g., community membership). The example is self-contained and uses `networkx` to create a graph similar to what you might export from Neo4j.

Sections:

1. Create a synthetic graph (Stochastic Block Model) with planted communities (ground-truth labels)
2. Compute adjacency and Laplacian matrices and inspect them
3. Build node feature matrix **X** and label vector **y** (with explicit explanation)
4. Train and evaluate a simple classifier (logistic regression)
5. Short notes: how to adapt when loading from Neo4j


In [ ]:
import numpy as np
import networkx as nx
import scipy.sparse as sp
from scipy.sparse import csgraph
from scipy.linalg import eigh
from sklearn.cluster import SpectralClustering
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

print('libraries imported')

## 1) Create a synthetic graph (Stochastic Block Model)

We build a graph with 3 communities. The planted community label will be our `y` (ground-truth).

In [ ]:
# parameters
n_communities = 3
sizes = [50, 50, 50]  # nodes per community
p_in = 0.08  # probability of edges within community
p_out = 0.005  # probability of edges between communities

# create block probability matrix
p = [[p_in if i==j else p_out for j in range(n_communities)] for i in range(n_communities)]

G = nx.stochastic_block_model(sizes, p, seed=42)

# relabel nodes to simple integers and record community labels
labels = []
for node, data in G.nodes(data=True):
    labels.append(data['block'])

n = G.number_of_nodes()
print('graph created: nodes=', n, 'edges=', G.number_of_edges())

# draw basic info
unique, counts = np.unique(labels, return_counts=True)
print('community sizes:', dict(zip(unique, counts)))

## 2) Adjacency and Laplacian matrices

The adjacency matrix **A** encodes edges. The (combinatorial) Laplacian is **L = D - A**, where **D** is the degree diagonal matrix. We also use the normalized Laplacian for spectral methods: **L_sym = I - D^{-1/2} A D^{-1/2}**.

In [ ]:
A = nx.to_scipy_sparse_matrix(G, nodelist=sorted(G.nodes()), format='csr', dtype=float)
D = sp.diags(A.sum(axis=1).A1)
L = D - A

# normalized Laplacian (symmetric)
L_norm = csgraph.laplacian(A, normed=True)

print('Adjacency shape:', A.shape)
print('Laplacian (combinatorial) shape:', L.shape)
print('Normalized Laplacian shape:', L_norm.shape)

# show a small dense view of the top-left corner for illustration
A_dense = A.todense()
print('\nTop-left corner of adjacency matrix:\n', A_dense[:6,:6])

### 2.1) Spectral decomposition (useful features)

Compute a few smallest non-zero eigenvectors of the normalized Laplacian — these provide *spectral embeddings* which are commonly used as features for nodes.

In [ ]:
k = 3  # number of eigenvectors/features to extract
# use scipy.linalg eigh on dense small matrix for clarity (convert normalized laplacian to dense)
w, v = eigh(L_norm.toarray())

# eigenvalues are sorted ascending; skip the first zero eigenvalue (connected components)
print('smallest eigenvalues:', w[:6])

# take next k eigenvectors (avoid the trivial first eigenvector)
spectral_feats = v[:, 1:k+1]
print('spectral_feats shape:', spectral_feats.shape)

## 3) Construct features X and labels y

**What is y?**
- In this demonstration we use *planted community membership* (the block id) as the label `y`.

**What is X?** (features derived from graph structure)
- Structural features (local): degree, clustering coefficient, pagerank
- Spectral features (global): first few non-trivial eigenvectors of normalized Laplacian

Combining structural + spectral features gives a concise node embedding that a simple classifier can use.

In [ ]:
# structural features
degrees = np.array([d for n, d in G.degree()])
clustering = np.array(list(nx.clustering(G).values()))
pr = np.array(list(nx.pagerank(G, alpha=0.85).values()))

# spectral_feats computed earlier

# assemble X
X = np.column_stack([degrees, clustering, pr, spectral_feats])
y = np.array(labels)

print('X shape:', X.shape)
print('y shape:', y.shape)
print('\nSample X (first 5 rows):\n', X[:5])
print('\nSample y (first 20):\n', y[:20])

## 4) Train a simple classifier to predict community membership

We split nodes into train/test sets and train a logistic regression (multi-class) to predict the planted community label using `X` features. This demonstrates the practical use of X and y.

In [ ]:
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, np.arange(len(y)), test_size=0.3, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=500, multi_class='multinomial')
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print('Accuracy:', accuracy_score(y_test, y_pred))
print('\nClassification report:\n', classification_report(y_test, y_pred))

### 4.1) Use Laplacian for unsupervised community detection (Spectral Clustering)

Spectral clustering uses the Laplacian eigenvectors as low-dimensional embeddings and then clusters them (e.g., k-means). This ties the Laplacian to real tasks.

In [ ]:
sc = SpectralClustering(n_clusters=3, affinity='precomputed', assign_labels='kmeans', random_state=42)
# affinity matrix can be adjacency
labels_sc = sc.fit_predict(A)
print('Spectral clustering - accuracy vs planted labels:', accuracy_score(y, labels_sc))

## 5) Notes — adapting this when loading graph from Neo4j

- Export nodes and relationships from Neo4j (CSV or use the `py2neo` / `neo4j` Python driver) and build a `networkx` graph: `nx.from_pandas_edgelist()` or `nx.from_edgelist()`.
- When constructing **X** and **y**:
  - `y` is typically a label you care about (community, churn, role). If no label exists, you can use unsupervised methods (spectral clustering, node2vec).
  - `X` comes from node properties (age, type, text embeddings) plus structural features (degree, centrality, spectral embeddings).
- The Laplacian is not just for display: its eigenvectors are powerful features (spectral embedding) and are used in spectral clustering, graph convolutional networks (as a basis), and graph signal processing.

### Example: load from Neo4j with `neo4j` driver

```python
# pseudocode
from neo4j import GraphDatabase
import pandas as pd
# query relationships
edges = session.run('MATCH (a)-[r]->(b) RETURN id(a) as a, id(b) as b')
edge_df = pd.DataFrame([dict(rec) for rec in edges])
G = nx.from_pandas_edgelist(edge_df, 'a', 'b')
# node properties
props = session.run('MATCH (n) RETURN id(n) as id, n.prop as prop')
# merge props into nodes
```
